# B-cell Epitope Analysis

Parameterised template — run via papermill, do not execute directly.

```bash
uv run python scripts/report.py --target ERCC1
```

In [ ]:
# papermill injects TARGET and INTERFACE_CSV when executing
TARGET = "ERCC1"
INTERFACE_CSV = ""  # path to antigen_interface.csv, or empty string to skip

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import seaborn as sns

# Locate repo root by walking up until pyproject.toml is found
_cwd = Path().resolve()
REPO_ROOT = next(p for p in [_cwd, *_cwd.parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(REPO_ROOT / "src"))

from epitope_pipeline.integration.combine import load_and_combine

target_dir = REPO_ROOT / "data" / TARGET
out_dir    = REPO_ROOT / "outputs" / TARGET
out_dir.mkdir(parents=True, exist_ok=True)

print(f"Target : {TARGET}")
print(f"Data   : {target_dir}")
print(f"Outputs: {out_dir}")

In [ ]:
df = load_and_combine(target_dir, out_dir=out_dir)
print('Sequence length:', len(df))
df.head()

In [ ]:
import pandas as pd

interface_df = None
if INTERFACE_CSV:
    interface_df = pd.read_csv(INTERFACE_CSV).rename(
        columns={'position': 'res_id', 'amino_acid': 'residue'}
    )
    print(f'Loaded interface: {len(interface_df)} residues')
else:
    print('No interface CSV provided — interface panel will be skipped')

In [ ]:
df.to_csv(out_dir / 'combined_scores.csv', index=False)
print('Saved:', out_dir / 'combined_scores.csv')

for col in ['is_epitope_bepipred', 'is_epitope_discotope', 'is_epitope_graphbepi', 'is_epitope_AND']:
    if col in df.columns:
        n = df[col].sum()
        print(f'  {col}: {n} / {len(df)} ({100*n/len(df):.1f}%)')

In [ ]:
sns.set_style('whitegrid')
fig, ax = plt.subplots(figsize=(14, 4))

ax.plot(df['res_id'], df['bepipred_score'],
        color='mediumpurple', linewidth=2, label='Epitope Score')
ax.axhline(y=0.5, color='gray', linestyle='--', linewidth=1, label='Threshold (0.50)')
ax.fill_between(df['res_id'], 0, df['bepipred_score'],
                where=df['is_epitope_bepipred'],
                alpha=0.8, color='lavender', label='Predicted Linear Epitope')

ax.set_xlabel('Residue Position', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title(f'BepiPred 3.0: Linear B-cell Epitope Prediction — {TARGET}', fontsize=14, fontweight='bold')
ax.set_ylim(-0.05, max(0.6, df['bepipred_score'].max() * 1.1))
ax.legend()
plt.tight_layout()
fig.savefig(out_dir / 'bepipred_profile.png', dpi=300)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))

ax.plot(df['res_id'], df['discotope_score'],
        color='green', linewidth=2, label='Epitope Score')
ax.axhline(y=0.9, color='gray', linestyle='--', linewidth=1, label='Threshold (0.90)')
ax.fill_between(df['res_id'], 0, df['discotope_score'],
                where=df['is_epitope_discotope'],
                alpha=0.4, color='mediumseagreen', label='Predicted Conformational Epitope')

ax.set_xlabel('Residue Position', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title(f'DiscoTope 3.0: Conformational B-cell Epitope Prediction — {TARGET}', fontsize=14, fontweight='bold')
ax.set_ylim(-0.1, max(3.5, df['discotope_score'].max() * 1.1))
ax.legend()
plt.tight_layout()
fig.savefig(out_dir / 'discotope_profile.png', dpi=300)
plt.show()

In [ ]:
if 'graphbepi_score' in df.columns and df['graphbepi_score'].any():
    fig, ax = plt.subplots(figsize=(14, 4))

    ax.plot(df['res_id'], df['graphbepi_score'],
            color='steelblue', linewidth=2, label='Epitope Score')
    ax.axhline(y=0.1763, color='gray', linestyle='--', linewidth=1, label='Threshold (0.1763)')
    ax.fill_between(df['res_id'], 0, df['graphbepi_score'],
                    where=df['is_epitope_graphbepi'],
                    alpha=0.4, color='lightsteelblue', label='Predicted Conformational Epitope')

    ax.set_xlabel('Residue Position', fontsize=12)
    ax.set_ylabel('Score', fontsize=12)
    ax.set_title(f'GraphBepi: Conformational B-cell Epitope Prediction — {TARGET}', fontsize=14, fontweight='bold')
    ax.set_ylim(-0.05, max(0.4, df['graphbepi_score'].max() * 1.1))
    ax.legend()
    plt.tight_layout()
    fig.savefig(out_dir / 'graphbepi_profile.png', dpi=300)
    plt.show()
else:
    print('GraphBepi scores not available for this target — skipping plot.')

In [ ]:
if interface_df is not None:
    import matplotlib.patches as mpatches
    contact_colors = {'direct': '#E24B4A', 'adjacent': '#378ADD'}
    merged = df[['res_id']].merge(interface_df[['res_id', 'contact_type']], on='res_id', how='left')
    colors = merged['contact_type'].map(contact_colors).fillna('#D3D1C7')

    fig, ax = plt.subplots(figsize=(14, 1.5))
    ax.bar(merged['res_id'], 0.4, color=colors.tolist(), width=1.0, align='center')
    ax.set_ylim(0, 1)
    ax.set_yticks([])
    ax.set_ylabel('Interface\nContact', fontsize=8, rotation=0, labelpad=45, va='center')
    ax.set_xlabel('Residue Position', fontsize=12)
    ax.set_title(f'Antibody-Antigen Interface Contacts — {TARGET}', fontsize=13, fontweight='bold')
    legend_handles = [
        mpatches.Patch(color='#E24B4A', label='Direct'),
        mpatches.Patch(color='#378ADD', label='Adjacent'),
        mpatches.Patch(color='#D3D1C7', label='None'),
    ]
    ax.legend(handles=legend_handles, fontsize=8)
    plt.tight_layout()
    plt.show()
else:
    print('No interface data — skipping interface panel')

In [ ]:
has_graphbepi = 'graphbepi_score' in df.columns and df['graphbepi_score'].any()
has_interface = interface_df is not None
base_panels = 3 if has_graphbepi else 2
n_panels = base_panels + (1 if has_interface else 0)

panel_heights = [4] * base_panels + ([1.5] if has_interface else [])

import matplotlib.patches as mpatches
sns.set_style('whitegrid')
fig = plt.figure(figsize=(18, sum(panel_heights)))
gs = fig.add_gridspec(n_panels, 1, hspace=0.5, height_ratios=panel_heights)

ax1 = fig.add_subplot(gs[0])
ax2 = fig.add_subplot(gs[1], sharex=ax1)

ax1.plot(df['res_id'], df['bepipred_score'], color='mediumpurple', linewidth=2, label='Score')
ax1.axhline(y=0.5, color='gray', linestyle='--', linewidth=1, label='Threshold (0.50)')
ax1.fill_between(df['res_id'], 0, df['bepipred_score'],
                 where=df['is_epitope_bepipred'], alpha=0.8, color='lavender', label='Epitope')
ax1.set_ylabel('BepiPred Score', fontsize=11, fontweight='bold')
ax1.set_title(f'Linear B-cell Epitopes (BepiPred 3.0) — {TARGET}', fontsize=13, fontweight='bold')
ax1.set_ylim(-0.05, max(0.65, df['bepipred_score'].max() * 1.1))
ax1.legend()

ax2.plot(df['res_id'], df['discotope_score'], color='green', linewidth=2, label='Score')
ax2.axhline(y=0.9, color='gray', linestyle='--', linewidth=1, label='Threshold (0.90)')
ax2.fill_between(df['res_id'], 0, df['discotope_score'],
                 where=df['is_epitope_discotope'], alpha=0.4, color='mediumseagreen', label='Epitope')
ax2.set_ylabel('DiscoTope Score', fontsize=11, fontweight='bold')
ax2.set_title(f'Conformational B-cell Epitopes (DiscoTope 3.0) — {TARGET}', fontsize=13, fontweight='bold')
if not has_graphbepi and not has_interface:
    ax2.set_xlabel('Residue Position', fontsize=11, fontweight='bold')
ax2.legend()

if has_graphbepi:
    ax3 = fig.add_subplot(gs[2], sharex=ax1)
    ax3.plot(df['res_id'], df['graphbepi_score'], color='steelblue', linewidth=2, label='Score')
    ax3.axhline(y=0.1763, color='gray', linestyle='--', linewidth=1, label='Threshold (0.1763)')
    ax3.fill_between(df['res_id'], 0, df['graphbepi_score'],
                     where=df['is_epitope_graphbepi'], alpha=0.4, color='lightsteelblue', label='Epitope')
    ax3.set_ylabel('GraphBepi Score', fontsize=11, fontweight='bold')
    ax3.set_title(f'Conformational B-cell Epitopes (GraphBepi) — {TARGET}', fontsize=13, fontweight='bold')
    if not has_interface:
        ax3.set_xlabel('Residue Position', fontsize=11, fontweight='bold')
    ax3.legend()

if has_interface:
    ax_iface = fig.add_subplot(gs[base_panels], sharex=ax1)
    contact_colors = {'direct': '#E24B4A', 'adjacent': '#378ADD'}
    merged = df[['res_id']].merge(interface_df[['res_id', 'contact_type']], on='res_id', how='left')
    colors = merged['contact_type'].map(contact_colors).fillna('#D3D1C7')
    ax_iface.bar(merged['res_id'], 0.4, color=colors.tolist(), width=1.0, align='center')
    ax_iface.set_ylim(0, 1)
    ax_iface.set_yticks([])
    ax_iface.set_ylabel('Interface\nContact', fontsize=8, rotation=0, labelpad=45, va='center')
    ax_iface.set_xlabel('Residue Position', fontsize=11, fontweight='bold')
    legend_handles = [
        mpatches.Patch(color='#E24B4A', label='Direct'),
        mpatches.Patch(color='#378ADD', label='Adjacent'),
        mpatches.Patch(color='#D3D1C7', label='None'),
    ]
    ax_iface.legend(handles=legend_handles, fontsize=8)
    save_name = 'B_cell_epitope_with_interface.png'
else:
    save_name = 'B_cell_epitope_combined.png'

fig.savefig(out_dir / save_name, dpi=300, bbox_inches='tight')
print('Saved:', out_dir / save_name)
plt.show()

In [ ]:
epitopes = df[df['is_epitope_AND']]
print(f'{len(epitopes)} epitope residues (is_epitope_AND) out of {len(df)}')
display(epitopes)